# PatchTST — three lookback scales, walk-forward across six months (Google Colab)

Trains the **side models** for [BipowerQuant](https://github.com/NynsenFaber/BipowerQuant):
given a lookback, which triple barrier does the price touch first?

The target never moves — 60 bp barriers, a one-hour deadline, the same folds and
the same labels throughout. The one thing that changes is **how much history the
model reads before deciding**:

| Scale | Lookback | Against the one-hour deadline |
| :--- | ---: | :--- |
| `short` | 5 minutes | 1:12 |
| `mid` | 2 hours | 2:1 |
| `long` | 24 hours | 24:1 |

Three things are worth knowing before pressing Run all.

1. **One model per (scale, fold).** Each is trained only on months *before* its
   test month, so every number is a forecast rather than an in-sample fit. Three
   scales x three folds = nine models.
2. **The three scales are the same architecture.** `PatchTSTConfig.for_window`
   scales the patch length and stride with the lookback, so all three encode 37
   tokens and only the width of the patch embedding changes. A longer lookback
   therefore costs more compute per window but is not a bigger *model* in the
   sense that would make the comparison meaningless.
3. **A time budget, split across the scales.** Cell 8 measures real training
   throughput for each scale, projects the cost across every fold, and raises the
   training stride until the run fits its share. It will not let you start a
   thirty-hour job by accident.

### Before you press Run all

1. `Runtime -> Change runtime type -> Hardware accelerator: **T4 GPU**`
2. Run all. It pauses once, near the start, for Google Drive authorisation.

Bar caches and per-scale probability files are written to Drive as they are
produced, so a disconnect costs you at most the scale in flight.


## 0. Confirm the GPU

In [ ]:
!nvidia-smi -L || echo "No GPU attached — Runtime > Change runtime type > T4 GPU"

import torch
print("torch", torch.__version__, "| CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print("device:", props.name, f"| {props.total_memory / 1e9:.1f} GB",
          "| native bf16:", torch.cuda.get_device_capability(0)[0] >= 8)
    # Free, accuracy-neutral speed on Ampere and newer; a no-op on a T4.
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True


## 1. Pull the code

The branch below has to exist **on the remote** — push it before running this.

*No remote access?* Skip this cell, upload `sequence_matrix.py`, `patchtst_model.py`,
`patchtst_train.py`, `patchtst_folds.py`, `backtest.py`, `walkforward.py` and
`data_feeder.py` through the file browser into `/content/`, then run
`sys.path.insert(0, "/content")` instead.

In [ ]:
REPO_URL = "https://github.com/NynsenFaber/BipowerQuant.git"
BRANCH   = "long-window"
REPO_DIR = "/content/BipowerQuant"

import os, sys, subprocess

# xgboost/sklearn ship with Colab, but walkforward.py imports both and a
# missing one would only surface after the clone, so pin them here.
%pip install -q polars xgboost scikit-learn

if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, REPO_DIR],
                   check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=False)

sys.path.insert(0, f"{REPO_DIR}/python")

import glob, json, shutil, time, urllib.request, zipfile
import numpy as np

import patchtst_folds as pf
import sequence_matrix as seq
import walkforward as wf
from patchtst_model import PatchTSTClassifier, PatchTSTConfig, batch_size_for
from patchtst_train import TrainConfig, pick_device

print("code loaded from", REPO_DIR)


## 2. Configuration

`MONTHS` drives everything downstream. Six months gives three anchored folds at
`TRAIN_MONTHS = 3`; two months is enough to check the pipeline runs.

`SCALES` picks which lookbacks to train. Drop one to shorten the run — the
downstream backtest reads one probability file per scale and simply reports
whichever ones exist.

`TIME_BUDGET_HOURS` is the **total** across every scale, and it is enforced
rather than advisory. It is split evenly, and cell 8 raises each scale's training
stride until its share fits.


In [ ]:
# ---- data ----
SYMBOL    = "BTCUSDT"
MONTHS    = ["2026-01", "2026-02", "2026-03", "2026-04", "2026-05", "2026-06"]
USE_DRIVE = True                                   # cache bars + weights on Drive
DRIVE_DIR = "/content/drive/MyDrive/BipowerQuant"
DATA_DIR  = "/content/data"

# ---- problem definition (keep matched to python/sequence_matrix.py) ----
HORIZON  = 3600         # vertical barrier: decide within one hour
BARRIER  = 0.0060       # +/-60 bp horizontal barriers
CHANNELS = "raw"        # "raw" = log_return + ofi; "full" = the old 6 channels

# ---- the lookbacks under comparison ----
# Names from seq.WINDOW_SCALES: short = 300 s, mid = 7200 s, long = 86400 s.
# The patch geometry is derived from each, so nothing else here changes with it.
SCALES = ["short", "mid", "long"]

# ---- walk-forward ----
SCHEME       = "anchored"     # "anchored" | "rolling" | "holdout"
TRAIN_MONTHS = 3

# ---- model (identical across scales; only the patch geometry scales) ----
D_MODEL, N_HEADS, N_LAYERS, D_FF = 64, 4, 6, 128
DROPOUT, HEAD_DROPOUT = 0.2, 0.2
USE_SCALE_FEATURES = True
NORM = "batch"

# ---- training ----
EPOCHS, BATCH_SIZE = 12, 512  # batch is clamped down where a patch is 4,608 bars wide
LR, WEIGHT_DECAY   = 3e-4, 1e-4
PATIENCE           = 3
SEED               = 42
VAL_FRAC           = 0.10     # carved off the END of each fold's training months
VAL_STRIDE         = 8

# ---- the budget, TOTAL across every scale ----
TIME_BUDGET_HOURS = 9.0


## 3. Fetch the tape

Each month is downloaded, folded into 1-second bars, cached to Drive as a ~27 MB
`.npz`, and the multi-gigabyte CSV is **deleted immediately**. Six months of raw
CSV is ~52 GB and does not fit on a Colab disk; six months of bars is 165 MB.

Already-cached months are skipped, so a reconnect costs nothing.

In [ ]:
os.makedirs(DATA_DIR, exist_ok=True)
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    os.makedirs(DRIVE_DIR, exist_ok=True)
CACHE_DIR = DRIVE_DIR if USE_DRIVE else DATA_DIR

bar_paths = []
for month in MONTHS:
    cache = f"{CACHE_DIR}/bars_{SYMBOL}_{month}.npz"
    bar_paths.append(cache)
    if os.path.exists(cache):
        print(f"{month}: cached")
        continue

    url = (f"https://data.binance.vision/data/spot/monthly/trades/"
           f"{SYMBOL}/{SYMBOL}-trades-{month}.zip")
    zip_path = f"{DATA_DIR}/{SYMBOL}-trades-{month}.zip"
    started = time.perf_counter()
    print(f"{month}: downloading ...", end=" ", flush=True)
    urllib.request.urlretrieve(url, zip_path)
    with zipfile.ZipFile(zip_path) as archive:
        csv_name = archive.namelist()[0]
        archive.extract(csv_name, DATA_DIR)
    os.remove(zip_path)

    print("folding to bars ...", end=" ", flush=True)
    month_bars = seq.load_second_bars(f"{DATA_DIR}/{csv_name}", hours=None)
    seq.save_bars(month_bars, cache)
    os.remove(f"{DATA_DIR}/{csv_name}")   # never hold two months of CSV at once
    del month_bars
    print(f"done in {time.perf_counter() - started:.0f}s")

print(f"\n{len(bar_paths)} month(s) ready")


## 4. Splice and label

The months are spliced into one continuous 1-second grid — the market does not
restart between archives — and labelled with the triple barrier.

The labels do not depend on the lookback: which barrier the price touches first,
starting from a given bar, is a property of the *future*. So this runs once and
all three scales share it. What the lookback changes is only which bars can start
a window, and therefore how many windows there are.


In [ ]:
bars = seq.load_bar_caches(bar_paths)
meta = bars["meta"]
print(f"{meta['n_bars']:,} bars | {(meta['last_ts'] - meta['first_ts']) / 86400:.0f} days "
      f"| {meta['empty_seconds'] / meta['n_bars']:.1%} trade-less seconds")

side, _, defined = seq.triple_barrier(bars["price"], HORIZON, BARRIER)
resolved = (side != 0) & defined
print(f"barrier touched from {resolved.mean():.2%} of bars "
      f"| upper-first among those {(side[resolved] > 0).mean():.2%}")

print(f"\n{'scale':<8} {'lookback':>10} {'windows':>14} {'touched':>9}")
for scale in SCALES:
    window = seq.window_for(scale)
    starts = seq.valid_window_starts(bars["price"].size, window, HORIZON)
    touched = resolved[starts + window - 1]
    print(f"{scale:<8} {window:>9,}s {starts.size:>14,} {touched.mean():>8.2%}")


## 5. Folds

One model per (scale, fold), each trained only on months *before* its test month.
Nothing is carried between folds or between scales — reusing weights would leak a
later month into an earlier month's forecast, and reusing them across scales
would leak a lookback the model was not trained at.

Training keeps only windows where a barrier is actually touched, since a window
with no side has nothing to teach a side model. **Test scores every window**,
resolved or not: whether a window is worth trading is the *gate's* decision, taken
downstream in `walkforward.py`, and the gate needs a side for anything it lets
through.

Note how the training count falls as the lookback grows. It is not the barrier
resolving less often; it is the purge. Training rows are cut `window + horizon`
before each boundary, and at a 24-hour lookback that gap is a full day of the
month either side.


In [ ]:
device = pick_device()
folds  = wf.build_folds(bars["ts"], SCHEME, TRAIN_MONTHS)

print(f"device: {device}\n")
for scale in SCALES:
    window = seq.window_for(scale)
    cfg    = PatchTSTConfig.for_window(window, n_channels=len(seq.CHANNEL_SETS[CHANNELS]))
    starts = seq.valid_window_starts(bars["price"].size, window, HORIZON)
    touched = resolved[starts + window - 1]
    purge  = window + HORIZON - 1

    print(f"{scale} — lookback {window:,}s | patch {cfg.patch_len} stride {cfg.stride} "
          f"-> {cfg.num_patches} tokens | batch {batch_size_for(cfg, BATCH_SIZE)} | "
          f"{PatchTSTClassifier(cfg).n_parameters():,} params")
    print(f"  {'fold':<26} {'train':>12} {'val':>10} {'test (all)':>12}")
    for f in folds:
        tr, va, te = pf.fold_rows(f, starts, touched, purge, VAL_FRAC)
        print(f"  {f.name:<26} {tr.size:>12,} {va.size:>10,} {te.size:>12,}")
    print()


## 6. Train, inside the time budget

`train_folds` runs a throughput probe first — the *real* training step, same
autocast and gradient clipping, on a throwaway copy of the model — then picks the
smallest stride whose projected wall time across every fold fits that scale's
share of `TIME_BUDGET_HOURS`, and only then starts training.

Windows overlap by all but one of their bars, so striding discards far less
information than it discards rows, which is what makes this safe to decide
automatically. The projection ignores early stopping, so it is an upper bound.

Expect the stride the budget picks to **rise with the lookback**: a 24-hour window
gathers 288x as many bars per training example as a 5-minute one, and its patch
embedding is 288x wider. That is the honest cost of the longer lookback, and it is
worth reading the chosen strides as part of the result rather than as plumbing.

Each scale is written to Drive as it finishes, so an interrupted run keeps
whatever it has already earned. Re-running skips scales already on disk.


In [ ]:
per_scale_budget = TIME_BUDGET_HOURS / max(len(SCALES), 1)
probabilities, run_meta = {}, {}

for scale in SCALES:
    out = f"{CACHE_DIR}/patchtst_probs_{scale}.npz"
    if os.path.exists(out):
        print(f"{scale}: already trained -> {out}")
        stored = np.load(out)
        probabilities[scale] = {k: stored[k] for k in stored.files}
        continue

    window = seq.window_for(scale)
    print(f"\n{'#' * 78}\n# {scale}: {window:,}-second lookback, "
          f"budget {per_scale_budget:.1f} h\n{'#' * 78}")

    probs, meta_scale = pf.train_folds(
        bars, folds,
        window=window, horizon=HORIZON, barrier=BARRIER, channel_set=CHANNELS,
        config=PatchTSTConfig.for_window(
            window, n_channels=len(seq.CHANNEL_SETS[CHANNELS]),
            d_model=D_MODEL, n_heads=N_HEADS, n_layers=N_LAYERS, d_ff=D_FF,
            dropout=DROPOUT, head_dropout=HEAD_DROPOUT,
            use_scale_features=USE_SCALE_FEATURES, norm=NORM,
        ),
        train_config=TrainConfig(
            epochs=EPOCHS, batch_size=BATCH_SIZE, lr=LR, weight_decay=WEIGHT_DECAY,
            patience=PATIENCE, seed=SEED, amp=True,
        ),
        val_stride=VAL_STRIDE, val_frac=VAL_FRAC,
        time_budget_hours=per_scale_budget,
        device=device, checkpoint_dir="/content",
    )

    probabilities[scale], run_meta[scale] = probs, meta_scale
    np.savez_compressed(out, **probs)
    with open(f"{CACHE_DIR}/patchtst_folds_{scale}.json", "w") as fh:
        json.dump(meta_scale, fh, indent=2, default=float)
    print(f"wrote {out}")

print(f"\n{'scale':<8} {'fold':<26} {'test ROC-AUC':>13} {'best epoch':>11} {'stride':>7}")
for scale, meta_scale in run_meta.items():
    for name, row in meta_scale["folds"].items():
        print(f"{scale:<8} {name:<26} {row['roc_auc']:>13.4f} "
              f"{row['best_epoch']:>11} {meta_scale['train_stride']:>7}")


## 7. Export

One `patchtst_probs_<scale>.npz` per lookback, each holding one probability array
per fold, keyed by the fold name `walkforward.build_folds` generates. Download
them, put them beside your bar caches, and the local backtest scores every scale
through the identical execution model the tabular models go through:

```bash
cd python
for scale in short mid long; do
  python walkforward.py --bars ../data/bars_6m.npz --window-scale $scale \
                        --train-stride 5 \
                        --patchtst-probs ../weights/patchtst_probs_$scale.npz \
                        --out ../data/wf_$scale.json
done
python make_window_figures.py --results ../data/wf_short.json ../data/wf_mid.json ../data/wf_long.json
```

Read the ROC-AUC above as a *diagnostic*, not the result. It is measured only on
the windows where a barrier resolved, which is not a population you can select
into at decision time. The number that means something is what the backtest
reports on every window the gate lets through — and, across the three files, how
it moves with the lookback.


In [ ]:
written = []
for scale in SCALES:
    for name in (f"patchtst_probs_{scale}.npz", f"patchtst_folds_{scale}.json"):
        src = f"{CACHE_DIR}/{name}"
        if not os.path.exists(src):
            continue
        if src != f"/content/{name}":
            shutil.copy(src, f"/content/{name}")
        written.append(name)
        print(f"{name}  ({os.path.getsize(src) / 1e6:.1f} MB)")

if USE_DRIVE:
    for ckpt in glob.glob("/content/patchtst_*.pt"):
        shutil.copy(ckpt, DRIVE_DIR)
    print(f"checkpoints copied to {DRIVE_DIR}")

from google.colab import files
for name in written:
    files.download(f"/content/{name}")
